# Stitching / fusion spatialdata image elements

This notebook demonstrates how to use the `multiview_stitcher` package to stitch / fuse image elements in a `SpatialData` object. The example uses a subset of the CosMx NSCLC dataset, which can be downloaded from the `spatialdata-io` repository.

Dependencies:
- spatialdata
- spatialdata-io
- napari-spatialdata
- multiview_stitcher > 0.1.49

In [ ]:
from spatialdata_io import cosmx
from napari_spatialdata import Interactive

In [ ]:
# load sdata object
# example datasets: https://brukerspatialbiology.com/products/cosmx-spatial-molecular-imager/ffpe-dataset/

sdata = cosmx("/Users/albertm/Downloads/Lung13_data/Lung13-Flat_files_and_images")

In [ ]:
interactive = Interactive(sdata)
interactive.run()

# Fusing SpatialData images with multiview-stitcher

This function converts spatialdata image elements into multiview-stitcher `SpatialImage` objects,
fuses them, and returns the result as a new spatialdata element.

**Notes**:
- the spatialdata element's transformation to the target coordinate system is extracted as an affine matrix and passed to multiview-stitcher as the `transform_key`.
- After fusion, the fused image coordinates (spacing + origin) are encoded back into a spatialdata
  `Scale` + `Translation` transformation.
- Supports both single-scale (`DataArray`) and multiscale (`DataTree`) input images.
- Optional registration step before fusion.

In [ ]:
import numpy as np
import xarray as xr
from xarray import DataArray, DataTree

from spatialdata import SpatialData
from spatialdata.models import Image2DModel, Image3DModel, get_axes_names
from spatialdata.transformations import Scale, Translation, Sequence, get_transformation
from spatialdata.transformations.transformations import Identity

from multiview_stitcher import fusion, msi_utils, registration
from multiview_stitcher import spatial_image_utils as si_utils
from multiview_stitcher import param_utils

In [ ]:
def _spatialdata_image_to_sim(
    element: DataArray | DataTree,
    coordinate_system: str,
    transform_key: str,
    scale_level: str = "scale0",
) -> "si_utils.SpatialImage":
    """Convert a spatialdata image element to a multiview-stitcher SpatialImage.

    The spatialdata transformation (to `coordinate_system`) is embedded as the
    `transform_key` affine so that multiview-stitcher can fuse / register the tiles
    in that coordinate system.

    Parameters
    ----------
    element:
        Spatialdata image element (DataArray or multiscale DataTree).
    coordinate_system:
        Target coordinate system to extract the transformation for.
    transform_key:
        Key under which the affine transform is stored in multiview-stitcher.
    scale_level:
        Resolution level to extract from a multiscale DataTree, e.g. "scale0",
        "scale1", etc. Ignored for single-scale DataArray inputs.
    """
    # ---- 1. Get the DataArray at the requested resolution level ----
    if isinstance(element, DataTree):
        # ds = element[scale_level]
        NotImplementedError("Multiscale DataTree input not yet implemented in this function (but can be easily added).")
    else:
        da = element

    # ---- 2. Identify spatial axes ----
    axes = get_axes_names(da)  # e.g. ("c", "y", "x")
    spatial_axes = tuple(ax for ax in axes if ax in ("z", "y", "x"))

    # ---- 3. Extract the spatialdata affine matrix for the spatial dims ----
    transformation = get_transformation(element, to_coordinate_system=coordinate_system)
    # Returns an (ndim+1) x (ndim+1) homogeneous affine matrix
    affine_matrix = transformation.to_affine_matrix(
        input_axes=spatial_axes,
        output_axes=spatial_axes,
    )

    # ---- 4. Wrap as a multiview-stitcher xaffine DataArray ----
    # coords follow multiview-stitcher convention: ["z","y","x"][-ndim:] + ["1"]
    axis_labels = list(spatial_axes) + ["1"]
    xaffine = xr.DataArray(
        affine_matrix,
        dims=["x_in", "x_out"],
        coords={"x_in": axis_labels, "x_out": axis_labels},
    )

    # ---- 5. Intrinsic coordinates: translation (origin) and scale (spacing) ----
    # Extract the first coordinate of each spatial dim as the translation so that
    # the SpatialImage's intrinsic coordinates match those of the DataArray.
    translation = {
        ax: float(da.coords[ax][0]) if ax in da.coords else 0.0
        for ax in spatial_axes
    }
    scale = {
        ax: float(da.coords[ax][1] - da.coords[ax][0])
        if ax in da.coords and da.sizes[ax] > 1
        else 1.0
        for ax in spatial_axes
    }

    # ---- 6. Channel coordinates ----
    c_coords = da.coords["c"].values.tolist() if "c" in da.coords else None

    # ---- 7. Build the SpatialImage ----
    sim = si_utils.get_sim_from_array(
        da.data,
        dims=list(axes),
        scale=scale,
        translation=translation,
        transform_key=transform_key,
        affine=xaffine,
        c_coords=c_coords,
    )
    return sim


def _sim_to_spatialdata_image(
    fused_sim,
    coordinate_system: str,
    is_3d: bool,
) -> DataArray:
    """Convert a fused multiview-stitcher SpatialImage back to a spatialdata image element.

    The physical coordinates stored in the SpatialImage (origin + spacing) are
    encoded as a Scale + Translation spatialdata transformation.

    Note:
    - Further linear transformations need to be implemented still
    """
    spatial_dims = si_utils.get_spatial_dims_from_sim(fused_sim)
    origin  = si_utils.get_origin_from_sim(fused_sim)    # {dim: float}
    spacing = si_utils.get_spacing_from_sim(fused_sim)   # {dim: float}

    scale_values       = [spacing[ax] for ax in spatial_dims]
    translation_values = [origin[ax]  for ax in spatial_dims]

    t = Sequence([
        Scale(scale_values, axes=tuple(spatial_dims)),
        Translation(translation_values, axes=tuple(spatial_dims)),
    ])

    # multiview-stitcher always outputs a 't' dim; drop it if singleton
    if "t" in fused_sim.dims and fused_sim.sizes["t"] == 1:
        fused_sim = si_utils.sim_sel_coords(fused_sim, {"t": 0})

    c_coords = fused_sim.coords["c"].values.tolist() if "c" in fused_sim.dims else None
    dims     = tuple(fused_sim.dims)

    Model = Image3DModel if is_3d else Image2DModel
    return Model.parse(
        fused_sim.data,
        dims=dims,
        c_coords=c_coords,
        transformations={coordinate_system: t},
    )


def fuse_spatialdata_images(
    sdata: SpatialData,
    image_names: list[str],
    coordinate_system: str = "global",
    transform_key: str = "stage_metadata",
    scale_level: str = "scale0",
    output_spacing: dict[str, float] | None = None,
    do_align: bool = False,
    reg_channel: str | None = None,
    output_name: str = "fused",
) -> SpatialData:
    """Fuse spatialdata image elements using multiview-stitcher.

    Parameters
    ----------
    sdata:
        Input SpatialData object.
    image_names:
        Names of the image elements in `sdata.images` to fuse.
    coordinate_system:
        Target coordinate system for fusion (transformations are extracted
        relative to this system).
    transform_key:
        Key under which the tile transforms are stored in multiview-stitcher.
    scale_level:
        Resolution level to extract from multiscale images, e.g. "scale0" (full
        resolution), "scale1" (first downsampled level), etc.
    output_spacing:
        Desired pixel spacing of the fused output for each spatial dimension,
        e.g. ``{"y": 2.0, "x": 2.0}``. If None, the spacing of the first input
        tile (at the chosen scale level) is used.
    do_align:
        If True, run multiview-stitcher's pairwise alignment before fusion.
    reg_channel:
        Channel name to use for alignment (required if `do_align=True`).
    output_name:
        Name of the fused image in the returned SpatialData.

    Returns
    -------
    SpatialData with a single image element named `output_name`.
    """
    if do_align and reg_channel is None:
        raise ValueError("`reg_channel` must be provided when `do_align=True`.")

    # ---- Convert all spatialdata images to multiview-stitcher SpatialImages ----
    sims = [
        _spatialdata_image_to_sim(
            sdata.images[name], coordinate_system, transform_key, scale_level=scale_level
        )
        for name in image_names
    ]

    fuse_key = transform_key

    # ---- Optional alignment ----
    if do_align:
        registered_key = f"{transform_key}_registered"
        msims = [msi_utils.get_msim_from_sim(sim, scale_factors=[]) for sim in sims]
        registration.register(
            msims,
            reg_channel=reg_channel,
            transform_key=transform_key,
            new_transform_key=registered_key,
        )
        # Propagate the registered transforms back to the flat sims
        for sim, msim in zip(sims, msims):
            xaffine = si_utils.get_affine_from_sim(
                msi_utils.get_sim_from_msim(msim), transform_key=registered_key
            )
            si_utils.set_sim_affine(sim, xaffine, transform_key=registered_key)
        fuse_key = registered_key

    # ---- Fuse ----
    fused_sim = fusion.fuse(
        sims,
        transform_key=fuse_key,
        output_spacing=output_spacing,  # None → use spacing of first input tile
    )

    # ---- Convert back to spatialdata ----
    spatial_dims = si_utils.get_spatial_dims_from_sim(fused_sim)
    is_3d = "z" in spatial_dims

    fused_element = _sim_to_spatialdata_image(fused_sim, coordinate_system, is_3d)
    return SpatialData(images={output_name: fused_element})


In [ ]:
# Images to stitch are in the "global_only_image" coordinate system
image_names = [
    name for name in sdata.images
    if "global_only_image" in get_transformation(sdata.images[name], get_all=True)
]
print(f"Found {len(image_names)} images in 'global_only_image': {image_names}")


In [ ]:
result = fuse_spatialdata_images(
    sdata,
    image_names=image_names,
    coordinate_system="global",
    do_align=False,
    output_name="fused",
    output_spacing={"y": 18, "x": 18},
)

print(result)
fused_img = result.images["fused"]
print("Fused image shape:", fused_img.shape)
print("Fused image dims: ", fused_img.dims)


In [ ]:
# add to input sdata object

sdata['test'] = result.images['fused']

In [ ]:
interactive = Interactive(sdata)
interactive.run()